In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/diplomacy_lstm/diplomacy-deception-detection')
UPDATE_ZIP = Path('/content/diplomacy-phase5-update.zip')
print('Drive project:', PROJECT_ROOT)
assert PROJECT_ROOT.exists(), 'Copy your existing Phase 4 project to this location or edit PROJECT_ROOT.'


In [ ]:
import zipfile
if UPDATE_ZIP.exists():
    with zipfile.ZipFile(UPDATE_ZIP) as z:
        members=z.namelist()
        assert all(not name.startswith('/') and '..' not in Path(name).parts for name in members)
        assert all(not name.startswith(('data/','artifacts/')) for name in members), 'Update ZIP must not overwrite old data/artifacts.'
        z.extractall(PROJECT_ROOT)
    print('Phase 5 source/config/notebook/docs extracted into Drive project')
else:
    print('No update ZIP found; expecting Phase 5 files already present in Drive project')
assert (PROJECT_ROOT/'src/diplomacy/training/phase5_runner.py').exists()
assert all((PROJECT_ROOT/'data'/f'{split}.jsonl').exists() for split in ['train','validation','test'])
assert (PROJECT_ROOT/'artifacts/models/neural/phase_4/hierarchical_lstm/actual_lie/best_model.pt').exists(), 'Missing Phase 4 actual_lie baseline checkpoint'
assert (PROJECT_ROOT/'artifacts/models/neural/phase_4/hierarchical_lstm/suspected_lie/best_model.pt').exists(), 'Missing Phase 4 suspected_lie baseline checkpoint'


In [ ]:
import importlib.util,subprocess,sys,torch
required = {'transformers':'transformers>=4.40,<5.0', 'yaml':'PyYAML>=6.0', 'sklearn':'scikit-learn>=1.3'}
missing=[spec for module,spec in required.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
sys.path.insert(0,str(PROJECT_ROOT/'src'))
from diplomacy.training.phase5_runner import (
    ARCHITECTURES,TASKS,run_training,run_evaluation,load_trained,
    save_parity,verify_parity,compare_historical,predict_conversation)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)
if DEVICE=='cuda':
    print('GPU:',torch.cuda.get_device_name(0))
    print('GPU memory GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))


## Run settings — choose one or all eight experiments

In [ ]:
# Start with one combination; choose all only after testing setup.
SELECTED_ARCHITECTURES = ['hierarchical_lstm']
SELECTED_TASKS = ['actual_lie']

# For all eight (uncomment the two lines below):
# SELECTED_ARCHITECTURES = list(ARCHITECTURES)
# SELECTED_TASKS = list(TASKS)

ACTION = 'train'  # 'train' or 'evaluate'
RESUME = True
EPOCHS_OVERRIDE = None  # Uses the experiment YAML's epochs unless overridden
RUN_FINAL_TEST = False  # Only True AFTER all checkpoint decisions are final

print('Selected:',[(a,t) for a in SELECTED_ARCHITECTURES for t in SELECTED_TASKS])


## Train selected experiments sequentially

In [ ]:
training_results={}
if ACTION=='train':
    for arch in SELECTED_ARCHITECTURES:
        for task in SELECTED_TASKS:
            print('TRAIN:',arch,task)
            training_results[(arch,task)]=run_training(
                PROJECT_ROOT,arch,task,device=DEVICE,
                resume=RESUME,epochs_override=EPOCHS_OVERRIDE)
            print('Best validation:',training_results[(arch,task)]['best_validation_macro_f1'])
            if DEVICE=='cuda':torch.cuda.empty_cache()
print('Completed in this session:',len(training_results))


## Evaluate VALIDATION from best checkpoints

In [ ]:
validation_results={}
for arch in SELECTED_ARCHITECTURES:
    for task in SELECTED_TASKS:
        try:
            score,rows=run_evaluation(PROJECT_ROOT,arch,task,split='validation',device=DEVICE)
            validation_results[(arch,task)]=score
            print('VALIDATION',arch,task,'macro_f1=',round(score['macro_f1'],5),'lie_f1=',round(score['lie_f1'],5),'n=',score['n_samples'])
        except FileNotFoundError as exc:
            print('Not trained yet:',exc)


## Final TEST evaluation (one-time only after all validation choices are finalized)

In [ ]:
test_results={}
if RUN_FINAL_TEST:
    for arch in SELECTED_ARCHITECTURES:
        for task in SELECTED_TASKS:
            score,rows=run_evaluation(PROJECT_ROOT,arch,task,split='test',device=DEVICE)
            test_results[(arch,task)]=score
            print('TEST',arch,task,'macro_f1=',round(score['macro_f1'],5),'lie_f1=',round(score['lie_f1'],5),'n=',score['n_samples'])
else:
    print('RUN_FINAL_TEST=False: no test-set evaluation, as intended before checkpoint selection')


## Save Colab reference predictions for later Windows CPU parity

In [ ]:
parity_paths=[]
for arch in SELECTED_ARCHITECTURES:
    for task in SELECTED_TASKS:
        try:
            path=save_parity(PROJECT_ROOT,arch,task)
            parity_paths.append(path)
            print('Saved parity reference:',path)
        except FileNotFoundError as exc:
            print('Checkpoint missing:',arch,task,exc)


## Compare actual results with previous phases

In [ ]:
import pandas as pd
for task in TASKS:
    rows=compare_historical(PROJECT_ROOT,task)
    print('Comparison:',task)
    df=pd.DataFrame(rows)
    display(df)
    dest=PROJECT_ROOT/'artifacts'/'phase_5'/'comparisons'
    dest.mkdir(parents=True,exist_ok=True)
    df.to_csv(dest/f'{task}_comparison.csv',index=False)
    print('Saved:',dest/f'{task}_comparison.csv')
